# Congrats!!!!

Hello, if you reached here you have sucessfully ran a jupyter notebook.
This means you already have some basic knowledge of your shell yay

if you followed our instructions to get here there should be a `.venv\` directory now. 

That directory contains the virtual enviorment that has the code that runs this mini website


You can even run the python enviorment yourself with 
```bash
./.venv/bin/python
```

# Why Venvs???

This seems like kind of a clanky start.  Some of you may be used to just installing everything in one giant blob.
This CAN WORK sometimes, but as we will see shortly there are many places it fails.

One of the most obvious reasons where this can not work is when a package requires 2 diffrent versions of a depency. This can happen for many reasons, a lot of the time the issue starts in some C code or even the hardware itself.


## Exercise: simple venv 

A lot of times we want to call fast C/C++ code from python. This lets us run programs that are signifcantly faster (offten by a factor of more than 50x) to anything that can be written in python itself. This can sometimes raise a few issues.

In this example we will build two small pieces of custom C code and run them from Python. One requires NumPy 1.26 and the other requires NumPy 2.0. They should not be built in the same environment because an environment can only have one installed version of NumPy at a time. The exact C details are not important here; the important part is deciding which dependencies belong together.

A virtual environment is an isolated Python installation with its own packages. Creating one does not activate it, and activating it does not install anything. Activation only changes your shell so commands such as `python` and `pip` point at that environment. This lets two environments on the same computer contain conflicting package versions without replacing each other.


To create a venv call
```bash
uv venv my_venv
```

We can then activate our venv. This makes `python` and package-install commands use that particular environment.

```bash
source my_venv/bin/activate
```

It is possible to deactivate it later by simply typing
```bash
deactivate
```

Once it is active, we can install the tools needed by that environment:
```bash
uv pip install "numpy==1.26" setuptools
```
or 
```bash
uv pip install "numpy==2.0" setuptools
```

Your task is to create **two** virtual environments. Put NumPy 1.26 and `setuptools` in one, and NumPy 2.0 and `setuptools` in the other. Think about when you need to activate and deactivate each environment so an install cannot accidentally go into the wrong one.


### Running C

It is relatively straightforward to build and call C code from Python. Often, you are not even going to notice that you are doing it.

But for today, we are going to make the process more explicit.

There are two build files in this directory:

- `build_numpy_126.py` builds `numpy_126_hello` and must be run with the Python environment containing NumPy 1.26.
- `build_numpy_20.py` builds `numpy_20_hello` and must be run with the Python environment containing NumPy 2.0.

Build each extension using the correct environment.

The build creates native `.so` files (or `.pyd` files on Windows) in this directory. Notice that the generated file is outside the virtual environment even though the environment's Python and NumPy were used to build it.

Because both files are together, we can also try importing each one with the wrong NumPy version and see the ABI checks reject it:

```bash
.venv-numpy-1.26/bin/python -c "import numpy_20_hello; numpy_20_hello.hello()"
.venv-numpy-2.0/bin/python -c "import numpy_126_hello; numpy_126_hello.hello()"
```

NumPy is extremely nice to us here: it includes checks that make many ABI incompatibilities fail in a relatively well-contained way instead of letting incorrect native code continue running.

In general, C/C++ packages made for Python are written with a lot of care so that Python users do not have to deal directly with the difficult parts of systems programming. Most of the time, you should never need to think about any of this—except when several things have gone horribly wrong.


# Saving Environments

It might be tempting to save the venv as the way we distribute code. We can commit it into our Git history. Then everyone could avoid needing to set up the environment. They would just use our perfect copy.

The core issue is that different computers have different hardware and C libraries. So the compiled C code would not work the same. Then the surrounding Python code can sometimes change slightly around that... and the entire thing just collapses.

For this reason, we usually encode all the packages we want to install in a `requirements.txt` or `pyproject.toml`. TOML is more commonly used by packages themselves for pip to work with them nicely, while `requirements.txt` is used by most papers, as can be seen [here](https://github.com/paperswithcode/releasing-research-code/tree/master).

This system is not perfect and has many flaws, as we will see later on. It can somewhat randomly stop working after a few months or years. Unfortunately, Python is a bit of a mess, and some of this just can't be helped. But it is generally possible to fix the breakages that do happen, and this is still a fairly good way to handle things.

When recording a requirements file, it is very important to note that **ORDER MATTERS**. It actually matters a lot with pip. UV fixes a lot of this, but not all of it. A good rule of thumb is to test the requirements file on a completely fresh install before shipping.

To install from a file, simply run:

```bash
uv pip install -r requirements.txt
```

For `pyproject.toml`, it is even easier. UV will simply use that file without needing to be told to do so.

The venv will be created in `.venv` and generally remain hidden.


## Exercise: Alpine Docker

For this exercise, we will introduce another tool: **Docker**.

Docker gives us a convenient way to package and run software in a controlled environment. Instead of depending on whatever happens to be installed on a particular machine, we can describe the environment our program should run in and recreate it elsewhere.

We would now like to ship the Python program from the previous exercise into a Docker container. First run the NumPy examples from this lesson's directory so the local NumPy 2.0 `.so` exists:

```bash
./run_numpys.sh
```

We would like to ship this code to a diffrent enviorment. Alpine linux specifically. We made a DockerFile to construct that enviorment for us. Unfortunatly it is badly made and will fail. To see why use 

```bash
docker build -t venv-exercise .
docker run --rm venv-exercise
```

This builds and runs the docker container for us. Usually containers are build once during devlopment. Then get reused for deployment.


In this example the image should build, but it will fail when it runs. There are many ways runing an incompatible binary can fail. In this case we will get a link error about some gnu function. This is because Alpine does not use glibc, but our numpy extension assumed that it could use glibc.

In the real world this would usually happens as a result of something else. Like something expecting CUDA to exist or diffrent oneapi versions etc. The build system would warn aginst this issues before they get into python code. As long as you use best practices (avoid copying venvs this way).

To fix the issue we need to build the file on the correct machine. Then it would work just fine (The C code is smart enough to know what to do)

In a Dockerfile, `COPY` transfers existing files exactly as they are. It does not reinstall Python or rebuild native code for the image. `RUN` executes commands while constructing the image, so it can install software and generate files for that environment.

## Your task

Change the Dockerfile so Python and the required build tools are installed during the image setup. Copy the portable project inputs, such as `requirements.txt`, `build_numpy_20.py`, and `numpy_20_hello.c`, instead of copying the generated `.so`. Then create the venv, install its requirements, and call `build_numpy_20.py` inside Docker. Keep `run.sh` and the `CMD` unchanged.

When the environment is built in the place where it will run, the same `docker run` command will test the extension and congratulate you.


# Docker images and background services

The previous exercise showed why copying an environment from one system to another can fail. Docker is also useful because someone else can prepare and test an environment once, publish it as an **image**, and let us run it without compiling or installing the application ourselves.

We will use [Ollama](https://ollama.com/) as a concrete example. Ollama is a server for running language models locally. Run the commands in this section in a terminal so you can get comfortable using Bash. First, download Ollama's prebuilt image:


```bash
docker pull ollama/ollama
docker images ollama/ollama
```


`docker pull` downloads the image. It does not compile Ollama on this computer, and the image is not yet a running program. We create a container from it and start the Ollama server with `docker run`:


```bash
docker run -d \
    --name ollama \
    --device /dev/dri \
    -e OLLAMA_VULKAN=1 \
    -p 11434:11434 \
    -v ollama:/root/.ollama \
    ollama/ollama
```


The options describe how this container should run:

- `-d` runs it in the background and immediately gives the terminal back to us.
- `--name ollama` gives the container a convenient name for later commands.
- `--device /dev/dri` gives the container access to the Linux GPU devices.
- `-e OLLAMA_VULKAN=1` tells Ollama to use its Vulkan GPU backend.
- `-p 11434:11434` connects port `11434` on our computer to the same port in the container.
- `-v ollama:/root/.ollama` stores downloaded models in a persistent Docker volume.

The image contains Ollama and its userspace dependencies. The physical GPU and its kernel driver still belong to the host. Docker gives the container controlled access to them; it does not put hardware into the image.

An image and a container are therefore different things:

```text
IMAGE -- docker run --> CONTAINER -- runs --> OLLAMA SERVER
```


## Inspect the background service

Because the server is running in the background, `docker ps` now has a natural purpose: it shows us which containers are currently running. `docker logs` lets us inspect output from a background container.


```bash
docker ps
docker logs ollama
```


Look in the logs for `library=Vulkan`. This confirms that Ollama found a GPU through Vulkan.

The server is inside the container, but the port mapping lets a program on the host contact it at `localhost:11434`:

```text
host: localhost:11434 -- port mapping --> container: Ollama server :11434
```


```bash
curl http://localhost:11434/api/tags
```


Here `curl` runs on the host while the Ollama server runs inside the container. On a fresh volume, the response contains an empty model list.

## Run a model inside the container

`docker exec` runs an additional command inside an already-running container. The following command asks the Ollama server to download and run `qwen3:0.6b`, a small language model of about 523 MB. The first run takes longer because the model must be downloaded; later runs reuse the copy in the `ollama` volume.


```bash
docker exec ollama ollama run qwen3:0.6b --think=false \
    "In one short sentence, explain why a Docker image is useful."
```


The model command finished, but the server continues running in the background. We did not install Ollama, configure a Python environment for it, or compile its inference engine. Docker supplied the prepared application environment, while the volume keeps the model available to the server.

You can also start an interactive conversation with:

```bash
docker exec -it ollama ollama run qwen3:0.6b
```

Now the API also reports the downloaded model:


```bash
curl http://localhost:11434/api/tags
```


## Stop and restart the container

Stopping a container ends its processes but does not delete the container. `docker ps` only shows running containers, while `docker ps -a` also shows stopped ones.


```bash
docker stop ollama
docker ps -a --filter name=ollama
```


We can start the same container again. Its configuration is unchanged, and its model is still stored in the volume.


```bash
docker start ollama
docker ps --filter name=ollama
```


When you are finished, you can remove the container:

```bash
docker rm -f ollama
docker images ollama/ollama
```

This deletes the container, but `docker images` shows that the `ollama/ollama` image still exists. It can be used to create another container. Images and containers have separate lifecycles.

If you also want to remove the image, run:

```bash
docker rmi ollama/ollama
```

The named `ollama` model volume is separate again and is not deleted by either command.


# Exercise: Reproduction

We will now run a project that is somewhat of a reporoduction of the Tiny Stories paper.
It can run full training in about an hour for a 1 milion parameters LLM. https://github.com/raymond-van/gpt-tinystories

Like most projects it contains a requirments.txt The author was even kind enough to give us an exact version of all the packages they use. Surely this means it would run right? right... 

Unfortunatly no it would not run easy as you will see when trying to run it yourself. 
to start with find a nice place on your computer and run

```bash
git clone https://github.com/raymond-van/gpt-tinystories.git
```

to clone the code onto your machine. then run
```bash
cd gpt-tinystories
```

to get into that project directory. 



There are no build instructions since it is assumed the person runing this is competent (like you).

Clearly it is ment for a new venv to be created by the users prefered method.

And then a simple

```bash
pip install -r reqiorments.txt 
```
we can use 
```bash
uv pip install -r reqiorments.txt 
```

which will save us a few minutes and would give us a nicer error

# Fixing broken projects
As you can see in the real world a lot of ML projects don't build the first time you try it. This can range from a few minutes of anoyance to even DAYS on paticularly nasty combinations (needing diffrent hardware and pypi moving forward can really be anoying). We are not evil so this project should be possible to fix


try and use the tools we gave you to run the project. here are some clues in order you would need them

python 3.11

setuptools<82

mkdir models


# Advanced OPTIONAL

The code would be runing on CPU since it was made with a CUDA machine in mind. 
This is fixable with a few minor modifications and by installing ipex. OR by upgrading the pytorch version to something more modern.

For code older than 2025 intel has unique per GPU extensions that need to be installed, so this is system specific and somewhat compilcated.

This task should probably only be attempted once you are familar with pytorch. so later in this course